# Proyecto — Data Stream Processor

## Contexto (extremadamente importante): 

Uno de las mayores virtudes de un repositorio en github es la poder volver sobre los cambios hechos. Uno puede volver sbre algún `commit` he iniciar el proceso desde ese punto. 

En otro ejemplo parecido, al crear una lista en python, dicha lista se modifica al agregar o borrar elementos y regresar a un estado anterior de la lista no es tan sencillo. La idea de este proyecto es poder emular dicho proceso y realizar una especie de lista con memoria para poder llevar algunos registros de manera adecuada.


### Objetivo

Construya un pequeño sistema para recibir y procesar registros de datos utilizando las clases `ArrayStack` y `ArrayQueue` proporcionadas por el curso. La idea central es poder llevar la información en orden para llevar los cambios de los registros de manera adecuada.

En el método `__init__` debe aparecer tres atributos:
- **Queue:** registros que han llegado pero todavía no han sido procesados.
- **Stack:** historial de cambios realizados, para poder deshacer los cambios más recientes.
- **Lista:** como se encuentran los registros actualmente

Las clases `ArrayStack` y `ArrayQueue` ya están implementadas. **No debe implementarlas nuevamente ni modificarlas.**

## 1. Registro de datos

Cada registro es una tupla de tres elementos:

```python
(sensor, variable, value)
```

Ejemplos:

```python
("S01", "temperature", 23.5)
("S02", "temperature", 25.1)
("S01", "humidity", 61.2)
```

Una combinación única `(sensor, variable)` identifica un dato dentro del estado actual.

## 2. Clase `DataProcessor`

Implemente:

```python
class DataProcessor:
    ...
```

Debe utilizar:

- un `ArrayQueue` para los registros pendientes;
- un `ArrayStack` para el historial de cambios;
- un `list` para mantener el estado actual.

### Restricción

No sustituya `ArrayQueue` o `ArrayStack` por `list`, `collections.deque` u otra estructura para realizar las funciones que corresponden a la Queue o al Stack.

La clase `DataProcessor` no debe imprimir resultados. Los métodos deben devolver los valores especificados. Las impresiones utilizadas para demostrar el funcionamiento deben realizarse en las celdas de prueba.

## 3. `add(record)`

Agrega un registro a la `ArrayQueue`.

```python
processor.add(("S01", "temperature", 23.5))
```

Requisitos:

- agrega el registro a la cola;
- **no procesa** el registro;
- conserva el orden de llegada.

No se requiere un valor de retorno.

Debe rechazar registros que no tengan exactamente tres componentes o cuyo `value` no sea numérico. El tipo concreto de excepción para estos errores puede ser elegido por el estudiante, pero debe documentarse y utilizarse consistentemente.

## 4. `process_next()`

Procesa el siguiente registro pendiente.

Debe:

1. obtener el siguiente registro de la Queue;
2. procesarlo;
3. actualizar el estado actual;
4. guardar en el Stack la información necesaria para poder deshacer exactamente ese cambio.

### FIFO

Si se ejecuta:

```python
add(A)
add(B)
add(C)
```

las llamadas sucesivas a `process_next()` deben devolver/procesar `A`, luego `B` y luego `C`.

### Actualización

Si se procesa:

```python
("S01", "temperature", 23.5)
```

en el estado se debe reflejar:

```python
('S01', 'temperature', 23.5)
```

Si después se procesa `("S01", "temperature", 27.0)`, el valor actual debe ser `27.0`.

### Historial

Se debe agregar al `Stack` respectivo

### Retorno

Debe devolver el registro que acaba de ser procesado.

### Queue vacía

Si no hay registros pendientes, debe producir `Empty, el error creado en el repositorio `Goodrich`

## 5. `undo()`

Deshace el último cambio realizado mediante `process_next()`.

Ejemplo:

```text
20 → 25 → 30
```

Después de un `undo()`:

```text
20 → 25
```

Después de otro:

```text
20
```

Los cambios deben deshacerse en orden LIFO.

### Dato creado por primera vez

Suponga que inicialmente no existe `('S01', 'temperature', x)`, después de procesar:

```python
("S01", "temperature", 23.5)
```

el dato existe. Si se ejecuta `undo()`, debe volver a **no existir**.

### Historial vacío

Si no hay cambios que deshacer, debe producir `Empty`.

No se requiere un valor de retorno.

## 6. `pending()`

Devuelve el número de registros que todavía esperan ser procesados.

Por ejemplo, después de:

```python
add(A)
add(B)
add(C)
```

`pending()` debe devolver `3`. Después de `process_next()`, debe devolver `2`.

## 7. `current_value(sensor, variable)`

Devuelve el valor actual asociado con una combinación de sensor y variable.

Ejemplo:

```python
current_value("S01", "temperature")
```

puede devolver `23.5`.

Si nunca se ha procesado un registro para esa combinación, debe producir `KeyError`.

# 8. Ejemplo completo

Considere:

```python
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)
```

Después de `add(A)`, `add(B)`, `add(C)`, la Queue contiene `A → B → C`.

Después de procesar A, el estado contiene:

```text
S01 / temperature → 20
```

Después de procesar B:

```text
S01 / temperature → 25
```

Después de procesar C:

```text
S01 / temperature → 25
S01 / humidity    → 60
```

Un `undo()` elimina el efecto de C. Otro `undo()` elimina el efecto de B. Otro `undo()` elimina el efecto de A y el estado vuelve a estar vacío.

# 9. Pruebas obligatorias

Incluya pruebas para, como mínimo:

1. `pending()` sobre un procesador vacío.
2. Agregar un registro.
3. Agregar varios registros.
4. Verificar procesamiento FIFO.
5. Procesar un registro.
6. Procesar varios registros.
7. Actualizar una variable existente.
8. Consultar el valor actual.
9. Realizar un `undo()`.
10. Realizar varios `undo()` consecutivos.
11. Procesar cuando la Queue está vacía.
12. Hacer `undo()` cuando el historial está vacío.
13. Deshacer la creación de un dato que antes no existía.
14. Hacer varios cambios sobre la misma variable.
15. Agregar un registro con formato incorrecto.
16. Agregar un registro cuyo valor no sea numérico.
17. Consultar un sensor/variable que nunca haya sido procesado.

# 10. Análisis de complejidad

Explique la complejidad temporal de:

- `add`
- `process_next`
- `undo`
- `pending`
- `current_value`

Justifique qué operaciones determinan cada complejidad e indique qué estructuras auxiliares utiliza el sistema.

# 11. Restricciones

1. Utilice las clases `ArrayStack` y `ArrayQueue` proporcionadas.
2. No las reemplace por `list`, `deque` u otra estructura equivalente.
3. No modifique las implementaciones proporcionadas.
4. La implementación debe estar contenida en `DataProcessor`.
5. Incluya las pruebas solicitadas.
6. Explique brevemente sus decisiones de diseño.

# 12. Bonus — `redo()`

Como extensión opcional, implemente:

```python
redo()
```

Después de un `undo()`, el sistema debe poder volver a aplicar el cambio que acaba de deshacerse.

Por ejemplo:

```text
20 → 25 → 30
undo()  → 20 → 25
redo()  → 20 → 25 → 30
```

El estudiante debe explicar qué estructuras utiliza para implementar `redo()` y por qué. No se proporciona la estrategia de implementación.


# 13. Entrega

La entrega debe contener:

- implementación completa de `DataProcessor`;
- pruebas solicitadas;
- explicación breve de las decisiones de diseño;
- análisis de complejidad temporal;
- si realiza el bonus, implementación y explicación de `redo()`.

In [3]:
class DataProcessor: 
    def __init__(self):
        self.queue = ArrayQueue()
        self.stack = ArrayStack()
        self.lista = []

    def add(self, record):
        if len(record) != 3:
            raise ValueError("El registro debe tener 3 elementos")
        record[2] + 0
        self.queue.enqueue(record)

    def process_next(self):
        record = self.queue.dequeue()
        indice = -1
        for i in range(len(self.lista)):
            if self.lista[i][0] == record[0] and self.lista[i][1] == record[1]:
                indice = i
        if indice == -1:
            self.stack.push((len(self.lista), None))
            self.lista.append(record)
        else:
            self.stack.push((indice, self.lista[indice]))
            self.lista[indice] = record
        return record

    def undo(self):
        cambio = self.stack.pop()
        if cambio[1] is None:
            self.lista.pop()
        else:
            self.lista[cambio[0]] = cambio[1]

    def pending(self):
        return len(self.queue)

    def current_value(self, sensor, variable):
        for i in range(len(self.lista)):
            if self.lista[i][0] == sensor and self.lista[i][1] == variable:
                return self.lista[i][2]
        raise KeyError((sensor, variable))

In [5]:
from goodrich.ch06.array_stack import ArrayStack
from goodrich.ch06.array_queue import ArrayQueue

In [7]:
print("=== INICIO DE PRUEBAS FALTANTES (1 a 9) ===")
A = ("S01", "temperature", 20)
B = ("S01", "temperature", 25)
C = ("S01", "humidity", 60)

# 1. pending() en un procesador vacio
p = DataProcessor()
if p.pending() == 0:
    print("Prueba 1 OK ->", p.pending())
else:
    print("Prueba 1 ERROR ->", p.pending())

# 2. Agregar un registro
p = DataProcessor()
p.add(A)
if p.pending() == 1:
    print("Prueba 2 OK ->", p.pending())
else:
    print("Prueba 2 ERROR ->", p.pending())

# 3. Agregar varios registros
p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)
if p.pending() == 3:
    print("Prueba 3 OK ->", p.pending())
else:
    print("Prueba 3 ERROR ->", p.pending())

# 4. Procesamiento FIFO
p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)
primero = p.process_next()
segundo = p.process_next()
tercero = p.process_next()
if primero == A and segundo == B and tercero == C:
    print("Prueba 4 OK ->", primero, segundo, tercero)
else:
    print("Prueba 4 ERROR ->", primero, segundo, tercero)

# 5. Procesar un registro
p = DataProcessor()
p.add(A)
procesado = p.process_next()
if procesado == A and p.pending() == 0 and p.current_value("S01", "temperature") == 20:
    print("Prueba 5 OK ->", procesado)
else:
    print("Prueba 5 ERROR ->", procesado, p.pending())

# 6. Procesar varios registros
p = DataProcessor()
p.add(A)
p.add(B)
p.add(C)
p.process_next()
p.process_next()
p.process_next()
if p.pending() == 0 and p.current_value("S01", "temperature") == 25 and p.current_value("S01", "humidity") == 60:
    print("Prueba 6 OK -> temperature 25, humidity 60")
else:
    print("Prueba 6 ERROR")

# 7. Actualizar una variable existente
p = DataProcessor()
p.add(("S01", "temperature", 23.5))
p.add(("S01", "temperature", 27.0))
p.process_next()
antes = p.current_value("S01", "temperature")
p.process_next()
despues = p.current_value("S01", "temperature")
if antes == 23.5 and despues == 27.0:
    print("Prueba 7 OK ->", antes, "cambio a", despues)
else:
    print("Prueba 7 ERROR ->", antes, despues)

# 8. Consultar el valor actual
p = DataProcessor()
p.add(("S01", "humidity", 61.2))
p.add(("S02", "temperature", 25.1))
p.process_next()
p.process_next()
if p.current_value("S01", "humidity") == 61.2 and p.current_value("S02", "temperature") == 25.1:
    print("Prueba 8 OK -> 61.2 y 25.1")
else:
    print("Prueba 8 ERROR")

# 9. Realizar un undo()
p = DataProcessor()
p.add(A)
p.add(B)
p.process_next()
p.process_next()
antes = p.current_value("S01", "temperature")
p.undo()
despues = p.current_value("S01", "temperature")
if antes == 25 and despues == 20:
    print("Prueba 9 OK ->", antes, "volvio a", despues)
else:
    print("Prueba 9 ERROR ->", antes, despues)

=== INICIO DE PRUEBAS FALTANTES (1 a 9) ===
Prueba 1 OK -> 0
Prueba 2 OK -> 1
Prueba 3 OK -> 3
Prueba 4 OK -> ('S01', 'temperature', 20) ('S01', 'temperature', 25) ('S01', 'humidity', 60)
Prueba 5 OK -> ('S01', 'temperature', 20)
Prueba 6 OK -> temperature 25, humidity 60
Prueba 7 OK -> 23.5 cambio a 27.0
Prueba 8 OK -> 61.2 y 25.1
Prueba 9 OK -> 25 volvio a 20


In [ ]:
from goodrich.exceptions import Empty

print("=== INICIO DE PRUEBAS FALTANTES (10 a 17) ===")

dp10 = DataProcessor()
dp10.add(("S01", "temp", 20))
dp10.add(("S01", "temp", 25))
dp10.add(("S02", "hum", 60))
dp10.process_next()
dp10.process_next()
dp10.process_next()
dp10.undo()  
dp10.undo()
print("10. Varios undo(): valor S01/temp tras 2 undo ->", dp10.current_value("S01", "temp"))  # Esperado: 20


dp11 = DataProcessor()
try:
    dp11.process_next()
except Empty as e:
    print("11. process_next() en Queue vacía -> Se capturó correctamente la excepción Empty:", e)

dp12 = DataProcessor()
try:
    dp12.undo()
except Empty as e:
    print("12. undo() en Stack vacío -> Se capturó correctamente la excepción Empty:", e)

dp13 = DataProcessor()
dp13.add(("S01", "temp", 23.5))
dp13.process_next()
print("13. Antes de undo -> Estado lista:", len(dp13.lista))  
dp13.undo()
print("13. Después de undo -> Estado lista:", len(dp13.lista))  

dp14 = DataProcessor()
dp14.add(("S01", "temp", 10))
dp14.add(("S01", "temp", 20))
dp14.add(("S01", "temp", 30))
dp14.process_next() 
dp14.process_next()  
dp14.process_next()  
print("14. Valor inicial:", dp14.current_value("S01", "temp"))  # 30
dp14.undo()
print("14. Tras 1er undo:", dp14.current_value("S01", "temp"))  # 20
dp14.undo()
print("14. Tras 2do undo:", dp14.current_value("S01", "temp"))  # 10

dp15 = DataProcessor()
try:
    dp15.add(("S01", "temp")) 
except ValueError as e:
    print("15. Formato incorrecto -> Se capturó ValueError:", e)

dp16 = DataProcessor()
try:
    dp16.add(("S01", "temp", "veintitrés"))  
except TypeError as e:
    print("16. Valor no numérico -> Se capturó TypeError:", e)

dp17 = DataProcessor()
try:
    dp17.current_value("S99", "presion")
except KeyError as e:
    print("17. Variable no procesada -> Se capturó KeyError para:", e)

print("=== TODAS LAS PRUEBAS EJECUTADAS CON ÉXITO ===")

## Decisiones de Diseño

Para la implementación de la clase `DataProcessor`, se tomaron las siguientes decisiones de diseño y estructura de datos:

1. **Estructura del Historial en el `ArrayStack` (`(índice, registro_anterior)`):**
   * En el historial se almacenan tuplas con la forma `(índice, registro_anterior)`.
   * **Justificación:** Guardar la posición exacta del registro dentro de `self.lista` permite que la operación `undo()` se ejecute en tiempo constante \\(O(1)\\), evitando realizar búsquedas secuenciales adicionales al momento de deshacer un cambio.

2. **Representación de elementos nuevos con `None`:**
   * Cuando un registro se procesa por primera vez (no existía antes en `self.lista`), se guarda en el historial la tupla `(posición_de_inserción, None)`.
   * **Justificación:** El valor `None` sirve como centinela explícito para indicar la ausencia previa del dato. Durante un `undo()`, si se detecta `None`, la función ejecuta `self.lista.pop()` para eliminar el elemento recién creado.

3. **Manejo de Excepciones:**
   * **`ValueError` en `add()`:** Se utiliza para validar que la entrada sea una tupla/lista de exactamente 3 elementos.
   * **`TypeError` en `add()`:** Valida que la medición sea de tipo numérico (`int` o `float`).
   * **`Empty` en `process_next()` y `undo()`:** Se reutiliza directamente la excepción nativa de `ArrayQueue` y `ArrayStack`.
   * **`KeyError` en `current_value()`:** Notifica la ausencia de la combinación `(sensor, variable)` consultada.

```

In [ ]:
## Análisis de Complejidad Temporal

A continuación se detalla la complejidad asintótica de cada método de `DataProcessor` en notación Big-O, donde $n$ representa la cantidad de registros en `self.lista` y $m$ la cantidad de elementos pendientes en `self.queue`:

* **`add(record)` $\rightarrow O(1)$ amortizado:**
  * *Justificación:* Ejecuta validaciones simples $O(1)$ y llama a `self.queue.enqueue(record)`. Gracias al redimensionamiento dinámico por duplicación de la cola circular `ArrayQueue`, la inserción tiene un costo amortizado de $O(1)$.

* **`process_next()` $\rightarrow O(n)$:**
  * *Justificación:* Desencolar de la cola toma $O(1)$ amortizado. Sin embargo, verificar si la combinación `(sensor, variable)` ya existe requiere recorrer linealmente `self.lista` de tamaño $n$, lo que toma $O(n)$ en el peor caso. Las operaciones posteriores sobre `self.lista` y `self.stack` toman $O(1)$ amortizado.

* **`undo()` $\rightarrow O(1)$ amortizado:**
  * *Justificación:* Desapilar con `self.stack.pop()` es $O(1)$ amortizado. Como la tupla almacenada contiene el índice directo en `self.lista`, la restauración o la eliminación del elemento con `self.lista.pop()` toma $O(1)$.

* **`pending()` $\rightarrow O(1)$:**
  * *Justificación:* Consulta directamente la longitud de la cola con `len(self.queue)`, operación de costo constante $O(1)$.

* **`current_value(sensor, variable)` $\rightarrow O(n)$:**
  * *Justificación:* Realiza una búsqueda secuencial sobre `self.lista` de tamaño $n$. En el peor caso (elemento al final o inexistente), recorre los $n$ registros.

---

### Estructuras Auxiliares Utilizadas

1. **`ArrayQueue` (Cola Circular):** Mantiene el orden FIFO de los registros recibidos con operaciones $O(1)$ amortizadas.
2. **`ArrayStack` (Pila LIFO):** Permite deshacer cambios en orden LIFO con tiempo $O(1)$ amortizado.
3. **`list` (Arreglo Dinámico):** Almacena el estado actual de los registros procesados.